# Bangla Daraz ABSA Pipeline
### End-to-End Sentiment & Hierarchical Aspect-Based Sentiment Analysis (TF-IDF + PyTorch BiLSTM)
---


## 1. Environment & Setup
Initialize project directories, import dependencies, and set execution paths.


In [10]:
%load_ext autoreload
%autoreload 2

import sys, os, time
sys.path.insert(0, os.path.abspath("."))

from src.config import ensure_dirs, SENTIMENT_LABELS, ALL_ASPECTS
from src.data_processing import load_data, get_sentiment_split, get_aspect_split
from src.features import build_tfidf, BanglaVocab, create_dataloader, SentimentLSTM
from src.train import train_sentiment, train_aspects, train_polarities, train_lstm_sentiment, train_lstm_aspects
from src.predict import predict_hierarchical, predict_sentiment
from src.evaluate import save_confusion_matrix, save_metrics_summary_json, save_model_comparison_csv
from src.persistence import save_artifacts, save_polarity_artifacts, load_artifacts, load_polarity_artifacts

ensure_dirs()
START = time.time()
print("Environment ready")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Environment ready


## 2. Data Loading & Preprocessing
Load raw reviews (`annotated_bangla.csv`), clean Bangla text, parse labels into targets, and export `clean_annotated_bangla.csv`.


In [11]:
df = load_data()
Xs_tr, Xs_te, ys_tr, ys_te = get_sentiment_split(df)
Xa_tr, Xa_te, ya_tr, ya_te = get_aspect_split(df)
print(f"Loaded {len(df):,} reviews | train={len(Xs_tr):,} test={len(Xs_te):,}")


Loaded 2,016 reviews | train=1,612 test=404


## 3. TF-IDF Sentiment Classifier
Train 3-class Sentiment Classifier (`Positive`, `Negative`, `Neutral`) using Word + Character TF-IDF FeatureUnion and balanced Logistic Regression.


In [12]:
s_model, s_vec, s_metrics = train_sentiment(Xs_tr, ys_tr, Xs_te, ys_te)
save_artifacts("sentiment", s_model, s_vec, model_type="tfidf")
save_confusion_matrix(ys_te, s_metrics["predictions"], SENTIMENT_LABELS,
                      "Sentiment (TF-IDF)", "sentiment_tfidf.png")
print(f"Acc={s_metrics['accuracy']:.4f}  Macro-F1={s_metrics['macro_f1']:.4f}  Weighted-F1={s_metrics['weighted_f1']:.4f}")


Acc=0.9158  Macro-F1=0.8490  Weighted-F1=0.9197


## 4. TF-IDF Aspect Detection & Dedicated Polarity Models
Train multi-label OneVsRest aspect classifier and dedicated binary polarity classifiers for each aspect.


In [13]:
a_model, a_vec, a_bin, a_metrics = train_aspects(Xa_tr, ya_tr, Xa_te, ya_te)
save_artifacts("issue", a_model, a_vec, a_bin, model_type="tfidf")

pol_models = train_polarities(df)
save_polarity_artifacts(pol_models, model_type="tfidf")
print(f"Aspects Micro-F1={a_metrics['micro_f1']:.4f}  Macro-F1={a_metrics['macro_f1']:.4f}  Hamming-Loss={a_metrics['hamming_loss']:.4f}")


Aspects Micro-F1=0.9364  Macro-F1=0.8038  Hamming-Loss=0.0356


## 5. Vocabulary & PyTorch DataLoaders
Build Bangla vocabulary token mapping and prepare PyTorch DataLoaders for Sentiment and Multi-Label Aspect models.


In [14]:
from sklearn.preprocessing import MultiLabelBinarizer
from src.config import LSTM_BATCH_SIZE, LSTM_MAX_LENGTH

vocab = BanglaVocab()
vocab.fit(df["cleaned_text"].tolist())
print(f"Vocab size: {vocab.vocab_size}")

ys_tr_int = [SentimentLSTM.CLASSES.index(l) for l in ys_tr]
ys_te_int = [SentimentLSTM.CLASSES.index(l) for l in ys_te]
s_train = create_dataloader(Xs_tr, ys_tr_int, vocab, LSTM_MAX_LENGTH, LSTM_BATCH_SIZE)
s_test = create_dataloader(Xs_te, ys_te_int, vocab, LSTM_MAX_LENGTH, LSTM_BATCH_SIZE, shuffle=False)

mlb = MultiLabelBinarizer(classes=ALL_ASPECTS)
ya_tr_bin = mlb.fit_transform(ya_tr)
ya_te_bin = mlb.transform(ya_te)
a_train = create_dataloader(Xa_tr, ya_tr_bin, vocab, LSTM_MAX_LENGTH, LSTM_BATCH_SIZE)
a_test = create_dataloader(Xa_te, ya_te_bin, vocab, LSTM_MAX_LENGTH, LSTM_BATCH_SIZE, shuffle=False)


Vocab size: 1631


## 6. PyTorch BiLSTM Sentiment & Aspect Models

Train 2-layer Bidirectional LSTM models for Sentiment Analysis and Multi-Label Aspect Detection.


In [15]:
s_lstm, s_metrics_l = train_lstm_sentiment(s_train, s_test, vocab.vocab_size, ys_te.tolist())
save_artifacts("sentiment", s_lstm, model_type="lstm", vocab=vocab)
save_confusion_matrix(ys_te, s_metrics_l["predictions"], SENTIMENT_LABELS,
                      "Sentiment (LSTM)", "sentiment_lstm.png")

a_lstm, a_bin_l, a_metrics_l = train_lstm_aspects(a_train, a_test, vocab.vocab_size, ya_te_bin, mlb)
save_artifacts("issue", a_lstm, binarizer=a_bin_l, model_type="lstm", vocab=vocab)

print(f"LSTM Sent Acc={s_metrics_l['accuracy']:.4f} | Aspect Micro-F1={a_metrics_l['micro_f1']:.4f}")


    Training BiLSTM Sentiment Model...
      Epoch [01/10] - Loss: 0.6843
      Epoch [02/10] - Loss: 0.4206
      Epoch [03/10] - Loss: 0.3189
      Epoch [04/10] - Loss: 0.2765
      Epoch [05/10] - Loss: 0.2364
      Epoch [06/10] - Loss: 0.2211
      Epoch [07/10] - Loss: 0.1861
      Epoch [08/10] - Loss: 0.1793
      Epoch [09/10] - Loss: 0.1538
      Epoch [10/10] - Loss: 0.1406
    Training BiLSTM Multi-Label Aspect Model...
      Epoch [01/10] - Loss: 0.3735
      Epoch [02/10] - Loss: 0.2643
      Epoch [03/10] - Loss: 0.2048
      Epoch [04/10] - Loss: 0.1860
      Epoch [05/10] - Loss: 0.1705
      Epoch [06/10] - Loss: 0.1874
      Epoch [07/10] - Loss: 0.1664
      Epoch [08/10] - Loss: 0.1418
      Epoch [09/10] - Loss: 0.1322
      Epoch [10/10] - Loss: 0.1278
    Training BiLSTM Polarity [Product Quality]...
      Epoch [01/10] - Loss: 0.3857
      Epoch [02/10] - Loss: 0.1681
      Epoch [03/10] - Loss: 0.1232
      Epoch [04/10] - Loss: 0.0918
      Epoch [05/10] - L

## 7. Export Benchmark Comparison & Metrics Summary
Export comprehensive benchmark table (`model_comparison.csv`) and evaluation summary (`metrics_summary.json`).


In [16]:
rows = [
    {"Task": "Sentiment Analysis", "Model": "TF-IDF + LogReg",
     "Accuracy": round(s_metrics["accuracy"], 4), "Macro F1": round(s_metrics["macro_f1"], 4),
     "Weighted F1": round(s_metrics["weighted_f1"], 4), "Additional Metric": "N/A"},
    {"Task": "Sentiment Analysis", "Model": "LSTM",
     "Accuracy": round(s_metrics_l["accuracy"], 4), "Macro F1": round(s_metrics_l["macro_f1"], 4),
     "Weighted F1": round(s_metrics_l["weighted_f1"], 4), "Additional Metric": "N/A"},
    {"Task": "Aspect Detection", "Model": "TF-IDF + OvR",
     "Accuracy": round(1 - a_metrics["hamming_loss"], 4), "Macro F1": round(a_metrics["macro_f1"], 4),
     "Weighted F1": round(a_metrics["weighted_f1"], 4),
     "Additional Metric": f"Micro-F1: {a_metrics['micro_f1']:.4f}"},
    {"Task": "Aspect Detection", "Model": "LSTM",
     "Accuracy": round(1 - a_metrics_l["hamming_loss"], 4), "Macro F1": round(a_metrics_l["macro_f1"], 4),
     "Weighted F1": round(a_metrics_l["weighted_f1"], 4),
     "Additional Metric": f"Micro-F1: {a_metrics_l['micro_f1']:.4f}"},
]
for asp, e in pol_models.items():
    m = e["metrics"]
    rows.append({"Task": f"Polarity: {asp}", "Model": "TF-IDF + Binary LogReg",
                 "Accuracy": round(m["accuracy"], 4), "Macro F1": round(m["macro_f1"], 4),
                 "Weighted F1": round(m["weighted_f1"], 4),
                 "Additional Metric": f"n_train={e['n_train']}"})

save_model_comparison_csv(rows, "model_comparison.csv")
save_metrics_summary_json({
    "dataset": {"total": len(df)},
    "sentiment": {"tfidf": {k: v for k, v in s_metrics.items() if k != "predictions"},
                  "lstm": {k: v for k, v in s_metrics_l.items() if k != "predictions"}},
    "aspects": {"tfidf": {k: v for k, v in a_metrics.items() if k != "predictions"},
                "lstm": {k: v for k, v in a_metrics_l.items() if k != "predictions"}},
    "polarity": {"tfidf": {asp: {k: v for k, v in e["metrics"].items() if k != "predictions"} for asp, e in pol_models.items()}},
}, "metrics_summary.json")
print(f"Pipeline finished in {time.time()-START:.1f}s")


Pipeline finished in 365.0s


## 8. Live Inference Demo
Test Hierarchical ABSA inference on sample Bangla e-commerce customer reviews.


In [17]:
#TF-IDF
s_m, s_v, _, _ = load_artifacts("sentiment", "tfidf")
a_m, a_v, a_b, _ = load_artifacts("issue", "tfidf")
pol = load_polarity_artifacts("tfidf")

samples = [
    "প্রোডাক্ট ভালো কিন্তু ডেলিভারি দেরি হয়েছে।",
    "দাম অনেক বেশি, সেলার রিপ্লাই দেয় না।",
    "ব্যাটারি ভালো না একদমই।",
]
for text in samples:
    s = predict_sentiment(text, s_m, vectorizer=s_v)
    h = predict_hierarchical(text, a_m, pol, vectorizer=a_v, binarizer=a_b)
    print(f"\nReview: {text}")
    print(f"   Sentiment: {s['sentiment']} ({s['confidence']*100:.0f}%)")
    for d in h["aspect_details"]:
        print(f"   Aspect: {d['aspect']} | Polarity: {d['polarity']} ({d['confidence']*100:.0f}%)")



Review: প্রোডাক্ট ভালো কিন্তু ডেলিভারি দেরি হয়েছে।
   Sentiment: Neutral (89%)
   Aspect: Product Quality | Polarity: Positive (83%)
   Aspect: Delivery | Polarity: Negative (87%)

Review: দাম অনেক বেশি, সেলার রিপ্লাই দেয় না।
   Sentiment: Positive (47%)
   Aspect: Product Quality | Polarity: Positive (69%)
   Aspect: Price | Polarity: Positive (55%)
   Aspect: Seller Service | Polarity: Positive (65%)

Review: ব্যাটারি ভালো না একদমই।
   Sentiment: Negative (79%)
   Aspect: Product Quality | Polarity: Negative (91%)


In [18]:
# BiLSTM
s_m, _, _, vocab = load_artifacts("sentiment", "lstm")
a_m, _, a_b, _ = load_artifacts("issue", "lstm")
pol = load_polarity_artifacts("tfidf")

samples = [
    "প্রোডাক্ট ভালো কিন্তু ডেলিভারি দেরি হয়েছে।",
    "দাম অনেক বেশি, সেলার রিপ্লাই দেয় না।",
    "ব্যাটারি ভালো না একদমই।",
]
for text in samples:
    s = predict_sentiment(text, s_m, vocab=vocab)
    h = predict_hierarchical(text, a_m, pol, vocab=vocab, binarizer=a_b)
    print(f"\nReview: {text}")
    print(f"   Sentiment: {s['sentiment']} ({s['confidence']*100:.0f}%)")
    for d in h["aspect_details"]:
        print(f"   Aspect: {d['aspect']} | Polarity: {d['polarity']} ({d['confidence']*100:.0f}%)")



Review: প্রোডাক্ট ভালো কিন্তু ডেলিভারি দেরি হয়েছে।
   Sentiment: Neutral (92%)
   Aspect: Product Quality | Polarity: Positive (100%)
   Aspect: Delivery | Polarity: Positive (53%)

Review: দাম অনেক বেশি, সেলার রিপ্লাই দেয় না।
   Sentiment: Positive (95%)
   Aspect: Product Quality | Polarity: Positive (100%)
   Aspect: Price | Polarity: Positive (99%)

Review: ব্যাটারি ভালো না একদমই।
   Sentiment: Negative (98%)
   Aspect: Product Quality | Polarity: Negative (100%)
